Delection the Old Messages 

In [1]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import RemoveMessage  # to remove the premanant the delection messages in State
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY")

In [9]:
model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    api_key=GOOGLE_API_KEY
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [10]:
def chat(state: MessagesState):
    response= model.invoke(state['messages']) 
    return{'messages':[response]}
# delection the old messages 
def delete_old_messages(state: MessagesState):
    msgs= state["messages"]
    # if more than 10 messages , delete the earlieast 6
    if len(msgs) > 10:
        to_remove = msgs[:6]  # first 6 messages ko remove/delete kardo 
        return{
            "messages":[RemoveMessage(id=m.id) for m in to_remove] # us messages ki id ham send kar raha han jin ko delete karna ha.
        }
    return{}

In [11]:
builder = StateGraph(MessagesState)
builder.add_node("Chat",chat)
builder.add_node("cleanup",delete_old_messages)

builder.add_edge(START,"Chat")
builder.add_edge("Chat","cleanup")
builder.add_edge("cleanup","__end__")

In [12]:
graph = builder.compile(checkpointer=InMemorySaver())

In [ ]:
graph

In [13]:
config = {'configurable':{'thread_id':'chat-1'}}

In [14]:
graph.invoke({"messages":[{"role":"user", "content":"hi my name is ANAS"}]}, config)
graph.invoke({"messages":[{"role":"user", "content":"explain the TSMC?"}]}, config)
graph.invoke({"messages":[{"role":"user", "content":"about the chatgpt."}]}, config)
graph.invoke({"messages":[{"role":"user", "content":"about the gemini."}]}, config)
graph.invoke({"messages":[{"role":"user", "content":"what is my name."}]}, config)

{'messages': [HumanMessage(content='hi my name is ANAS', additional_kwargs={}, response_metadata={}, id='6fe6daf2-0375-4d08-ab03-c9278cc1a6ed'),
  AIMessage(content=[{'type': 'text', 'text': "Hello, Anas! It's nice to meet you. How can I help you today?", 'extras': {'signature': 'EoIFCv8EAQw51se1YIcwg5ldWd1lPKv3foA05BHtguGFmZPiFx+PfBCk7uyONcO+oRnbfT088EbJ2uyY6nz9ngxDIXmp+rxyDdVIg9kZOhaYLq6bl3w83KsQIOn5LRWYJkxzZvmsA2Wgl3IZRzQufhBD3FQoXhC8brmZwroL2epYb9XY/CtiVFwd/TvWtqC5N2/mOXbqq+nAjtq4GCcT9H70nDecfprOBclouDAVWV7DC2Qt5ZCpQKpIrX3CGiTNw5gZxyeGP2Ef3EivZxlmrCBRkrLo5fYmgL8vOeFUx3ATMIxsR7g9EcQnnOwKI5XYaLikberOQzNxbIvSs6icxqr1QHKcra1cfkpUCgG4Xnf5lvur27lRWBgqZN47OTnckyWWdGy8PDiLFSGDe9YoGsUxL6IZXv4eLM7oqkq12ZrPTzFarP5CS4kxiNcdnb2jgGmP+NoNVSPLM5oF1WzNTEoesdR+z0X//5Emj8DUu1FmnSPhuR0f+afnI9cyFcEY3svbFtJmEAQNYnM9mtqU+5DqCnVvLudPrtyfLIEzS8IAchd9bMAEUD2UdkKDkRvGZS2hZIT+ja0Vi3B3aQ4/e6QPxQxZdi59NenClno7dNxWzS1hxWFqv8NzegpSW1gucXnLNALCxSIvAVPEZ83rzjiBea8KiH20JCWJjbcR/cpkHRp1JvxEv4u/pIDmjBoIJ/6etLLZlE8eTbw

In [17]:
graph.invoke({"messages":[{"role":"user", "content":"what is about ASML?"}]}, config)

{'messages': [HumanMessage(content='about the gemini.', additional_kwargs={}, response_metadata={}, id='ff273acb-615e-47d4-828b-9f9f2a078749'),
  AIMessage(content=[{'type': 'text', 'text': 'Since we just talked about **TSMC** (the hardware) and **ChatGPT** (the rival), **Gemini** is the third piece of the puzzle.\n\n**Gemini** is the Artificial Intelligence developed by **Google** (specifically by Google DeepMind). It is Google’s direct answer to ChatGPT.\n\nHere is everything you need to know about it:\n\n### 1. What makes Gemini different?\nWhile ChatGPT was originally built for text and later "learned" to see and hear, Gemini was built to be **"Native Multimodal"** from the very start. \n*   This means Google designed it to understand text, images, video, audio, and computer code all at the same time, right from the first day of training.\n\n### 2. The Different "Sizes" of Gemini\nGoogle doesn’t just have one version of Gemini; they have a family of models designed for different ta

In [18]:
snap = graph.get_state(config)
print("store messages after cleanup:" ,len(snap.values["messages"]))

store messages after cleanup: 6
